# Lesson 03: Corner and Blob Detection — Python (Colab)

This notebook is divided into segments, each with an explanation followed by code. Run in order.


## 1. Setup and synthetic image

Imports libraries and builds a synthetic image containing flat regions, edges, corners/junctions, and blobs of different sizes to test feature detectors.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Synthetic image: corners, edges, blobs
H,W = 120,160
img = np.ones((H,W), np.float32)*40
img[20:90,20:70] = 220
img[40:100,90:140] = 220
yy,xx = np.mgrid[0:H,0:W]
for (cy,cx,r,val) in [(30,120,6,20),(80,40,12,20),(60,130,18,240)]:
    img[(yy-cy)**2 + (xx-cx)**2 < r*r] = val
img = np.clip(img,0,255)


## 2. Convolution helper and image gradients

Defines a basic 2D convolution and computes horizontal/vertical gradients (Ix, Iy) using Sobel kernels. Gradients reveal where and how intensity changes locally.


In [ ]:
def conv2(img,k):
    kh,kw=k.shape; ph,pw=kh//2,kw//2
    p=np.pad(img,((ph,ph),(pw,pw)),mode='edge'); out=np.zeros_like(img,np.float32)
    for i in range(img.shape[0]):
        for j in range(img.shape[1]):
            out[i,j]=np.sum(p[i:i+kh,j:j+kw]*k)
    return out

sobelx = np.array([[-1,0,1],[-2,0,2],[-1,0,1]],np.float32)
sobely = np.array([[-1,-2,-1],[0,0,0],[1,2,1]],np.float32)
Ix = conv2(img, sobelx); Iy = conv2(img, sobely)


## 3. Harris corner response

Builds the structure tensor components (Sxx, Syy, Sxy) from gradient products and computes the Harris response `R = det(M) - k*trace(M)^2`, which is strong when change occurs in two directions (corners).


In [ ]:
# Harris
Ixx,Iyy,Ixy = Ix*Ix, Iy*Iy, Ix*Iy
box = np.ones((3,3),np.float32)
Sxx,Syy,Sxy = conv2(Ixx,box), conv2(Iyy,box), conv2(Ixy,box)
k=0.05
det = Sxx*Syy - Sxy*Sxy
trace = Sxx+Syy
R = det - k*(trace**2)


## 4. Thresholding and non-maximum suppression (NMS)

Selects strong responses above a threshold and keeps only local maxima in a 3×3 neighborhood so each corner is represented once and better localized.


In [ ]:
# NMS
th = 0.1*R.max()
corners = np.zeros_like(R,bool)
for i in range(1,H-1):
    for j in range(1,W-1):
        if R[i,j]>th and R[i,j]==R[i-1:i+2,j-1:j+2].max():
            corners[i,j]=True


## 5. Multi-scale blob responses (LoG approximation)

Defines a Laplacian-of-Gaussian (LoG) kernel and computes responses across multiple scales. Blobs tend to respond most strongly when the filter scale matches their size.


In [ ]:
# Simple blob response: Laplacian of Gaussian approximation (LoG kernel) over scales
def log_kernel(sigma, size=None):
    if size is None: size = int(6*sigma+1)|1
    r = size//2; y,x = np.mgrid[-r:r+1,-r:r+1]
    s2=sigma*sigma
    k = (x*x + y*y - 2*s2)/(s2*s2) * np.exp(-(x*x+y*y)/(2*s2))
    return k - k.mean()

scales=[2,4,6,8,10]
responses=[conv2(img, log_kernel(s)) for s in scales]
stack = np.stack([np.abs(r) for r in responses], axis=0)
best = stack.max(axis=0)


## 6. Visualization

Shows the image, gradient magnitude, Harris response with detected corners, and LoG responses at different scales (including the max-over-scales view).


In [ ]:
fig,axs = plt.subplots(2,3,figsize=(13,7))
axs[0,0].imshow(img,cmap='gray'); axs[0,0].set_title('Image')
mag = np.sqrt(Ix*Ix+Iy*Iy)
axs[0,1].imshow(mag,cmap='magma'); axs[0,1].set_title('Gradient magnitude')
axs[0,2].imshow(R,cmap='RdBu_r'); axs[0,2].scatter(np.where(corners)[1],np.where(corners)[0],s=10,c='cyan'); axs[0,2].set_title('Harris R + NMS')
axs[1,0].imshow(np.abs(responses[1]),cmap='viridis'); axs[1,0].set_title('LoG response (σ=4)')
axs[1,1].imshow(np.abs(responses[3]),cmap='viridis'); axs[1,1].set_title('LoG response (σ=8)')
axs[1,2].imshow(best,cmap='viridis'); axs[1,2].set_title('Max over scales')
for a in axs.ravel(): a.axis('off')
plt.tight_layout(); plt.show()


## Experiments

- Adjust `k` and the threshold, or use Gaussian weighting in the structure tensor.
- Find local extrema across (x,y,σ) in `stack` for true scale-space blob detection; try different blob sizes.
